In [71]:
import pandas as pd
from scipy.stats import chi2_contingency
import numpy as np
import unicodedata


In [50]:
df1=pd.read_csv("../data/terrasses-autorisations.csv", sep=";")
df2=pd.read_csv("../data/dans-ma-rue.csv", sep=";")

In [51]:
df2['adresse']=df2['adresse'].str.split(',').str[0]

In [52]:
df2['adresse']

0                    16 rue Victor Duruy
1             31 Boulevard Saint-Jacques
2                          2 Rue d'Orsel
3                    63 Quai de la Seine
4          8 Boulevard de Bonne Nouvelle
                       ...              
1474280                     21 Rue Titon
1474281            16 Avenue Léon Bollée
1474282           236 Boulevard Voltaire
1474283                    2 Rue Emeriau
1474284            199 Rue du Chevaleret
Name: adresse, Length: 1474285, dtype: object

In [53]:
df2['adresse']= df2['adresse'].str.upper()

In [54]:
df2['adresse']=df2['adresse'].str.strip()
df1['adresse']=df1['adresse'].str.strip()

# Retirer les accents

In [55]:
def strip_accents(s):
    if pd.isna(s):
        return s
    nfkd = unicodedata.normalize('NFKD', s)
    return ''.join(c for c in nfkd if not unicodedata.combining(c))

df1['adresse'] = df1['adresse'].apply(strip_accents)
df2['adresse'] = df2['adresse'].apply(strip_accents)

# Collapse des espaces multiples en un seul espace

In [56]:
df1['adresse'] = df1['adresse'].str.replace(r'\s+', ' ', regex=True)
df2['adresse'] = df2['adresse'].str.replace(r'\s+', ' ', regex=True)

# Re-passer upper + strip apres ces transformations, sur les deux (idempotent, juste pour securiser)

In [57]:
df1['adresse'] = df1['adresse'].str.upper().str.strip()
df2['adresse'] = df2['adresse'].str.upper().str.strip()

In [58]:
# Verifier les virgules multiples dans l'adresse ORIGINALE de dans-ma-rue 
# -> necessite de relire la colonne brute puisque df2['adresse'] a deja ete splitee

In [59]:
tmp = pd.read_csv("../data/dans-ma-rue.csv", sep=";", usecols=["adresse"])
print(tmp['adresse'].str.count(',').value_counts())

adresse
1    1474251
2         31
0          3
Name: count, dtype: int64


# Mesure du recouvrement apres nettoyage

In [60]:
adresses_terrasses = set(df1['adresse'].dropna().unique())
adresses_dmr = set(df2['adresse'].dropna().unique())
communes = adresses_terrasses & adresses_dmr

print(f"Adresses uniques terrasses    : {len(adresses_terrasses)}")
print(f"Adresses uniques dans-ma-rue  : {len(adresses_dmr)}")
print(f"Adresses communes             : {len(communes)}")
print(f"Taux de recouvrement (cote terrasses) : {len(communes)/len(adresses_terrasses):.1%}")

Adresses uniques terrasses    : 14862
Adresses uniques dans-ma-rue  : 152698
Adresses communes             : 12021
Taux de recouvrement (cote terrasses) : 80.9%


In [61]:
df1['adresse'].duplicated().sum()

np.int64(9344)

In [62]:
df2['adresse'].duplicated().sum()

np.int64(1321587)

In [63]:
df1['periode_installation'].unique()

<StringArray>
[                nan,     'Toute l'année', 'du 11/04 Au 10/10',
 'du 01/10 Au 31/03', 'du 01/04 Au 30/09', 'du 15/03 Au 15/09',
 'du 31/12 Au 30/12', 'du 10/03 Au 10/09', 'du 31/05 Au 30/11']
Length: 9, dtype: str

In [64]:
df1.groupby('adresse').size()

adresse
- 143 BOULEVARD LEFEBVRE    1
0 RUE DU MONT CENIS         1
01 RUE DOMREMY              1
1 ALLEE DARIUS MILHAUD      1
1 AVENUE D'ITALIE           2
                           ..
RUE DU PELICAN              1
RUE FERDINAND DUVAL         1
RUE SAINT DOMINIQUE         1
RUE SURCOUF                 1
SSSS                        1
Length: 14862, dtype: int64

In [65]:
tailles = df1.groupby('adresse').size()
print((tailles > 1).sum())        # nombre d'adresses avec 2+ terrasses
print(tailles.value_counts())     # repartition complete : combien d'adresses ont 1, 2, 3... terrasses

5770
1     9092
2     3673
3     1253
4      502
5      185
6       88
7       36
8       22
10       4
9        4
11       2
15       1
Name: count, dtype: int64


In [66]:
# 1. Adresses avec une seule terrasse (attribution non ambigue)
tailles = df1.groupby('adresse').size()
adresses_uniques = tailles[tailles == 1].index

# 2. On filtre df1 sur ces adresses
df1_clean = df1[df1['adresse'].isin(adresses_uniques)]

# 3. Inner join avec df2
df_join = df1_clean.merge(df2, on='adresse', how='inner', suffixes=('_terrasse', '_signalement'))

print(df_join.shape)

(94785, 28)


In [73]:
# Tableau croisé : effectifs bruts
contingence = pd.crosstab(df_join['typologie'], df_join['type'])
print(contingence)

# Meme tableau mais en % par ligne, plus lisible pour comparer les typologies entre elles
print(pd.crosstab(df_join['typologie'], df_join['type'], normalize='index'))

# Test statistique
chi2, p, dof, expected = chi2_contingency(contingence)
print(f"p-value : {p}")

type                                                Activités commerciales et professionnelles  \
typologie                                                                                        
COMMERCE ACCESSOIRE                                                                          1   
CONTRE ETALAGE                                                                               0   
CONTRE TERRASSE ESTIVALE SUR PLACES ET TERRE-PLEIN                                          19   
CONTRE TERRASSE ESTIVALE SUR STATIONNEMENT                                                1256   
CONTRE TERRASSE ESTIVALE SUR TROTTOIR DÉSAXÉE P...                                           3   
CONTRE TERRASSE ESTIVALE SUR TROTTOIR FACE À LA...                                         188   
CONTRE TERRASSE SUR TROTTOIR                                                                22   
CONTRE TERRASSE SUR VOIE PIÉTONNE                                                           15   
CONTRE ÉTALAGE SUR P

In [74]:
# n = nombre total de signalements dans le tableau croise
n = contingence.sum().sum()

# r = nombre de typologies de terrasse differentes (lignes du tableau)
# c = nombre de types de signalement differents (colonnes du tableau)
r, c = contingence.shape

# formule du V de Cramer
cramers_v = (chi2 / (n * min(r - 1, c - 1))) ** 0.5

print(f"n (total signalements) : {n}")
print(f"r (typologies)         : {r}")
print(f"c (types signalement)  : {c}")
print(f"V de Cramer            : {cramers_v:.3f}")

n (total signalements) : 94774
r (typologies)         : 29
c (types signalement)  : 10
V de Cramer            : 0.075


In [ ]:
adresses_terrasses = set(df1['adresse'].dropna().unique())
df2['a_une_terrasse'] = df2['adresse'].isin(adresses_terrasses)

contingence_h1 = pd.crosstab(df2['a_une_terrasse'], df2['type'])
print(contingence_h1)

chi2_h1, p_h1, dof_h1, expected_h1 = chi2_contingency(contingence_h1)

n_h1 = contingence_h1.sum().sum()
r_h1, c_h1 = contingence_h1.shape
cramers_v_h1 = (chi2_h1 / (n_h1 * min(r_h1 - 1, c_h1 - 1))) ** 0.5

print(f"p-value H1     : {p_h1}")
print(f"V de Cramer H1 : {cramers_v_h1:.3f}")